# GLITCHSCOPE 边缘AI · 只读体验
四格依次检查、查询、计划预览、导出报告。不会初始化板卡或执行试验。

In [ ]:
from pathlib import Path
import sys,json,uuid
sys.dont_write_bytecode=True
KIT=Path.cwd().resolve()
if not (KIT/'edge_api.py').is_file():raise RuntimeError('请从边缘AI包根目录打开此Notebook')
sys.path.insert(0,str(KIT))
from edge_package import verify
print('固定文件检查:',verify(notebooks=True))
from edge_api import EdgeAI
from custom_ai_core import query,TARGETS
AI=EdgeAI()
print('模型:',AI.model.model_sha256)
print('三个目标分别建模；当前只做文件/软件操作，不访问FPGA。')


In [ ]:
choice=input('目标 1=B完整帧，2=STM载荷，3=STM业务 [1]：').strip() or '1'
if choice not in ('1','2','3'):raise ValueError('目标必须是1、2或3')
TARGET=TARGETS[int(choice)-1]
values=input('d,s,phase,width [3,-1,15,5]：').strip() or '3,-1,15,5'
parts=[int(x.strip()) for x in values.split(',')]
if len(parts)!=4:raise ValueError('请输入4个整数')
D,S,PHASE,WIDTH=parts
PREDICTION=AI.predict(query(d=D,s=S,phase=PHASE,width=WIDTH,target=TARGET))
print(json.dumps(PREDICTION,ensure_ascii=False,indent=2))
print('score是未校准分数，不是硬件可靠率；prediction为null表示仍需实测。')


In [ ]:
from edge_api import example_request
from custom_ai_core import canonical
BUDGET=int(input('预览总预算，含CONFIG和无扰动参照 [14]：').strip() or '14')
WORK=KIT.parent/'edge_ai_workspaces'/('preview_'+uuid.uuid4().hex[:12])
WORK.mkdir(parents=True,exist_ok=False)
PREVIEW=AI.compile(example_request(TARGET,D,S,BUDGET))
(WORK/'plan_preview.json').write_bytes(canonical(PREVIEW))
print('计划状态:',PREVIEW['status'],'错误:',PREVIEW.get('errors'))
print('仅预览，不授权硬件。完整计划:',WORK/'plan_preview.json')


In [ ]:
from edge_report import render
REPORT=render(WORK/'model_report')
print('模型与评估报告:',REPORT)
from IPython.display import display,HTML
from urllib.parse import quote
notebook_root=Path('/home/xilinx/jupyter_notebooks')
if notebook_root in REPORT.parents:
    url='/files/'+quote(REPORT.relative_to(notebook_root).as_posix())
    display(HTML('<a target="_blank" href="'+url+'">打开或下载只读报告</a>'))
print('可下载report.html在电脑打开；CSV/JSON在同目录。没有创建板端试验会话。')
